# 건축공학 Cowork 프로젝트 설계 — AE Cowork Design

**Introduction to Claude Cowork — L07~L10 대응 (건축공학 도메인 응용)**

이 노트북에서 다루는 내용:
1. 건축공학 Cowork 프로젝트의 Instructions 설계
2. 파일 작업 프롬프트 패턴 (Input → Transformation → Output)을 구조설계에 적용
3. 대규모 분석: 다수 부재의 병렬 구조 검토 시뮬레이션
4. 보고서 자동 생성 프롬프트 작성

> **목표**: 구조설계 사무소에서 Cowork를 활용하여 구조 검토, 보고서 생성, 도면 분석을 자동화하는 프로젝트를 설계합니다.

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
import asyncio
import json
import os
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

## §1. 건축공학 Cowork 프로젝트 Instructions 설계

실제 Cowork에서 "프로젝트"를 만들고 Instructions를 설정하는 것을 시뮬레이션합니다.
구조설계 사무소의 프로젝트에 최적화된 Instructions를 작성합니다.

In [ ]:
# ── 건축공학 프로젝트 Instructions ──────────────────────
AE_INSTRUCTIONS = """
# 건축공학 구조설계 프로젝트

## 프로젝트
- 명칭: 경희대학교 신공학관 증축
- 용도: 교육시설 (특수), 지상 8층 / 지하 2층
- 구조: RC 라멘 + 코어월 (커플링빔 연결)
- 규모: 연면적 12,000m², 기준층 1,500m²

## 적용 기준
- 구조설계: KDS 41 10 15 (건축물 구조기준), KDS 14 20 (콘크리트 구조)
- 내진설계: KDS 41 17 00, 내진등급 I (특), 지역계수 S=0.22g
- 하중조합: KDS 41 10 25

## 재료
- 콘크리트: fck = 30MPa (기둥/벽체), 27MPa (보/슬래브)
- 철근: fy = 400MPa (SD400), fwy = 400MPa (전단보강)

## 팀 & 연락처
- 백교수: 구조설계 총괄 (baekjw@khu.ac.kr)
- 김대리: 구조해석 담당
- 이과장: 시설팀 (발주처)
- 감리단: 최감리 단장

## 폴더 구조
- ./analysis/     ← 구조해석 결과 (MIDAS MGT 파일)
- ./calculations/  ← 구조계산서
- ./drawings/      ← 구조도면 (CAD)
- ./reports/       ← 출력 보고서
- ./meetings/      ← 설계회의 기록

## 출력 규칙
- 단위: SI (kN, mm, MPa, kN·m)
- 소수점: 응력 1자리, 변위 2자리, 비율 3자리
- 안전율: D/C ratio로 표기, NG 시 빨간색 강조
- 보고서: .docx 형식, 표지 포함
"""

# Instructions 확인 요청 (Cowork에서 "내가 여기서 어떻게 작업하는지 알려줘")
response = client.messages.create(
    model=MODEL,
    max_tokens=512,
    system=AE_INSTRUCTIONS,
    messages=[{"role": "user", "content": "내가 여기서 어떻게 작업하는지 알고 있는 것을 알려줘."}]
)
print("=== Cowork 컨텍스트 확인 ===")
print(response.content[0].text)

## §2. 파일 작업 프롬프트: 다수 부재 병렬 구조 검토

Cowork의 핵심 강점: **폴더의 모든 파일을 읽고** 병렬로 분석합니다.
여기서는 여러 RC 부재의 구조 검토를 서브에이전트가 동시에 수행하는 시나리오를 시뮬레이션합니다.

In [ ]:
# ── 구조 검토 스킬 ──────────────────────────────────────
STRUCTURAL_REVIEW_SKILL = """
# /structural-review 스킬

## 검토 절차
1. 입력 부재 정보 파싱
2. 휨 강도 검토: φMn vs Mu (φ=0.85)
3. 전단 강도 검토: φVn vs Vu (φ=0.75)
4. D/C ratio 계산 및 판정

## 출력 형식 (JSON)
{"member": "...", "flexure": {"Mu": N, "phiMn": N, "dc_ratio": N, "verdict": "OK/NG"},
 "shear": {"Vu": N, "phiVn": N, "dc_ratio": N, "verdict": "OK/NG"},
 "overall": "OK/NG/REVIEW"}
"""

# 다수 부재 데이터 (실제로는 구조해석 결과 파일에서 읽어올 데이터)
members = [
    {"id": "B1-2F", "type": "보", "b": 400, "d": 540, "fck": 27, "fy": 400,
     "As": 2027, "Mu": 280, "Vu": 150, "Av_s": 0.50},
    {"id": "B2-3F", "type": "보", "b": 350, "d": 490, "fck": 27, "fy": 400,
     "As": 1520, "Mu": 210, "Vu": 120, "Av_s": 0.40},
    {"id": "C1-1F", "type": "기둥", "b": 600, "d": 540, "fck": 30, "fy": 400,
     "As": 3040, "Mu": 450, "Vu": 220, "Av_s": 0.70},
    {"id": "W1-CORE", "type": "벽체", "b": 300, "d": 2700, "fck": 30, "fy": 400,
     "As": 5070, "Mu": 1800, "Vu": 500, "Av_s": 1.20},
]

async def review_member(member: dict) -> dict:
    """서브에이전트: 개별 부재를 독립적으로 구조 검토한다."""
    async_client = anthropic.AsyncAnthropic()
    
    system = AE_INSTRUCTIONS + "\n\n" + STRUCTURAL_REVIEW_SKILL
    
    response = await async_client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=system,
        messages=[{
            "role": "user",
            "content": (
                f"다음 부재의 구조 검토를 수행하세요.\n"
                f"부재: {member['id']} ({member['type']})\n"
                f"단면: {member['b']}x{member['d']}mm, fck={member['fck']}MPa, fy={member['fy']}MPa\n"
                f"철근: As={member['As']}mm², Av/s={member['Av_s']}mm²/mm\n"
                f"설계력: Mu={member['Mu']}kN·m, Vu={member['Vu']}kN\n"
                f"반드시 유효한 JSON만 출력하세요."
            )
        }]
    )
    try:
        result = json.loads(response.content[0].text)
    except json.JSONDecodeError:
        result = {"member": member["id"], "error": "JSON 파싱 실패", "raw": response.content[0].text[:200]}
    
    verdict = result.get("overall", "?")
    print(f"  {'✅' if verdict == 'OK' else '❌'} {member['id']}: {verdict}")
    return result

# 서브에이전트 병렬 실행
print(f"=== {len(members)}개 부재 병렬 구조 검토 시작 ===\n")
tasks = [review_member(m) for m in members]
results = await asyncio.gather(*tasks)

print(f"\n=== 전체 검토 결과 ===")
print(json.dumps(results, indent=2, ensure_ascii=False))

## §3. Cowork 프롬프트 설계: 건축공학 실전

실제 Cowork에서 사용할 프롬프트를 **Input → Transformation → Output** 패턴으로 설계합니다.

In [ ]:
# ── 건축공학 Cowork 프롬프트 라이브러리 ──────────────────
AE_PROMPTS = {
    "구조계산서 생성": {
        "input": "analysis/ 폴더의 MIDAS 해석 결과 파일들",
        "transformation": "각 부재별 휨/전단 검토, D/C ratio 산정, NG 부재 표시",
        "output": "calculations/ 폴더에 structural_calc.xlsx로 저장",
        "prompt": (
            "analysis/ 폴더의 구조해석 결과를 읽고, 모든 주요 부재(보, 기둥, 벽체)에 대해 "
            "KDS 14 20 기준으로 휨 및 전단 강도를 검토해줘. "
            "D/C ratio가 1.0을 초과하는 부재는 빨간색으로 표시하고, "
            "결과를 calculations/structural_calc.xlsx에 부재 유형별 시트로 저장해줘."
        )
    },
    "설계회의 보고서": {
        "input": "meetings/ 폴더의 최근 회의록들",
        "transformation": "결정사항 추출, 미결 이슈 정리, 다음 액션 아이템 도출",
        "output": "reports/ 폴더에 meeting_summary.docx로 저장",
        "prompt": (
            "meetings/ 폴더의 최근 3개 회의록을 읽고, 구조설계 관련 결정사항을 정리해줘. "
            "미결 이슈와 담당자별 액션 아이템을 표로 만들고, "
            "이과장에게 보낼 수 있는 형식으로 reports/meeting_summary.docx에 저장해줘."
        )
    },
    "배근 검토": {
        "input": "drawings/ 폴더의 구조도면 (PDF)",
        "transformation": "배근 간격/피복/겹이음 검토, KDS 기준 대비",
        "output": "reports/ 폴더에 rebar_review.docx로 저장",
        "prompt": (
            "drawings/ 폴더의 구조도면을 읽고, 주요 부재의 배근 상세를 검토해줘. "
            "최소 배근간격, 피복두께, 겹이음 길이를 KDS 14 20 기준과 비교하고, "
            "부적합 사항을 사진과 함께 reports/rebar_review.docx에 정리해줘."
        )
    }
}

# 프롬프트 라이브러리 출력
print("=== 건축공학 Cowork 프롬프트 라이브러리 ===\n")
for name, info in AE_PROMPTS.items():
    print(f"📋 {name}")
    print(f"   Input: {info['input']}")
    print(f"   Transform: {info['transformation']}")
    print(f"   Output: {info['output']}")
    print(f"   Prompt: {info['prompt'][:80]}...")
    print()

In [ ]:
# ── "구조계산서 생성" 프롬프트를 실제로 실행 (시뮬레이션) ──
prompt = AE_PROMPTS["구조계산서 생성"]["prompt"]

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=AE_INSTRUCTIONS + "\n\n" + STRUCTURAL_REVIEW_SKILL,
    messages=[{
        "role": "user",
        "content": (
            f"{prompt}\n\n"
            f"(시뮬레이션) 다음 부재 데이터를 사용하세요:\n"
            f"{json.dumps(members, ensure_ascii=False, indent=2)}"
        )
    }]
)
print("=== 구조계산서 생성 결과 (시뮬레이션) ===")
print(response.content[0].text)

## §4. 핵심 정리

- **Instructions 설계**: 프로젝트명, 적용 기준, 재료, 팀, 폴더 구조, 출력 규칙을 포함하면 Cowork가 건축공학 전문가처럼 작동한다.
- **파일 작업 프롬프트**: Input(해석 결과) → Transformation(구조 검토) → Output(계산서 파일) 패턴이 가장 효과적이다.
- **병렬 분석**: 다수 부재를 서브에이전트가 동시에 검토하면 시간과 정확도가 모두 향상된다. 각 부재가 독립적인 컨텍스트에서 분석되므로 간섭이 없다.
- **실전 적용**: 실제 Cowork에서 이 노트북의 Instructions와 프롬프트를 그대로 사용할 수 있다.

### Cowork 도입 5단계 (건축공학 버전)
1. **플러그인 설치** → 구조설계 플러그인 (또는 직접 제작)
2. **실행** → 하나의 구조계산서 생성 작업부터 시작
3. **스킬 제작** → 자주 쓰는 검토 워크플로를 스킬 파일로 저장
4. **예약** → "매주 금요일 주간 구조검토 보고서" 자동 생성
5. **공유** → 팀원과 스킬/플러그인 공유